# Laboratorio 10: ETL, EDA y Feature Engineering - Titanic Dataset

Bienvenido a este laboratorio avanzado de análisis de datos. En esta sesión, exploraremos el legendario dataset del Titanic para entender qué factores influyeron en la supervivencia de los pasajeros. Pasaremos por un flujo completo de Ciencia de Datos: desde la extracción de datos (ETL) y el análisis exploratorio (EDA) hasta una ingeniería de características (Feature Engineering) detallada, preparando la información para futuros modelos de aprendizaje automático.

## 📋 Tabla de Contenidos

1. [Configuración Inicial e Imports](#-imports)
2. [ETL: Extracción, Transformación y Carga](#-etl)
3. [EDA: Análisis Exploratorio de Datos](#-eda)
4. [Feature Engineering: Ingeniería de Características](#-feature-engineering)
5. [Preprocesamiento con Scikit-Learn](#-sklearn)

---

<a id='-imports'></a>
## 🔹 1: Configuración Inicial e Imports

En esta sección preparamos el entorno de trabajo importando las librerías necesarias para el manejo de datos, visualización y preprocesamiento. Configuramos los parámetros globales para garantizar gráficos consistentes y profesionales.

In [ ]:
#!pip install jupyter-black
%load_ext jupyter_black

In [ ]:
# import warnings
# import pandas as pd
# import numpy as np
# from matplotlib import pyplot as plt
# import seaborn as sns
# import pylab as plot
# from sklearn.preprocessing import LabelEncoder

# warnings.filterwarnings("ignore")
# warnings.filterwarnings("ignore", category=DeprecationWarning)

# %matplotlib inline

# # Configuración de visualización para Pandas
# pd.options.display.max_columns = 100

# # Parámetros de graficación
# params = {
#     "axes.labelsize": "large",
#     "xtick.labelsize": "x-large",
#     "legend.fontsize": 20,
#     "figure.dpi": 150,
#     "figure.figsize": [25, 7],
# }
# plot.rcParams.update(params)

In [ ]:
import warnings
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
import pylab as plot
from sklearn.preprocessing import LabelEncoder

# ==== FIX para evitar que Jupyter se caiga mostrando DataFrames ====

# Guardamos la función original
_old_repr_html_ = pd.DataFrame._repr_html_

def _safe_repr_html_(self):
    # Convertimos columnas a string para evitar errores de renderizado
    df2 = self.copy()
    df2.columns = df2.columns.astype(str)
    return _old_repr_html_(df2)

# Sobrescribimos el método de Pandas
pd.DataFrame._repr_html_ = _safe_repr_html_

# ==== FIN DEL FIX ====

# Warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=DeprecationWarning)

# Matplotlib inline
%matplotlib inline

# Configuración de visualización para Pandas
pd.options.display.max_columns = 100

# Parámetros de graficación
params = {
    "axes.labelsize": "large",
    "xtick.labelsize": "x-large",
    "legend.fontsize": 20,
    "figure.dpi": 150,
    "figure.figsize": [25, 7],
}
plot.rcParams.update(params)

---

<a id='-etl'></a>
## 🔹 2: ETL (Extracción, Transformación y Carga)

El primer paso consiste en cargar el conjunto de datos y realizar un diagnóstico inicial de su estructura, tipos de datos y calidad de la información (valores faltantes).

### 📝 2.1 Carga de Datos

Cargamos el archivo CSV que contiene los datos de entrenamiento del Titanic.

In [ ]:
# Cargar el training set
data = pd.read_csv("train.csv")

### 📝 2.2 Información Inicial del Dataset

Utilizamos métodos fundamentales de Pandas para obtener una visión general de la arquitectura de los datos.

In [ ]:
# Resumen técnico de las columnas y tipos de datos
data.info()

In [ ]:
# Muestra aleatoria para validación
print("\nMuestra aleatoria de 5 registros:")
data.sample(5)

In [ ]:
# Estadísticas descriptivas generales
print("\nResumen estadístico de variables numéricas:")
data.describe()

---

<a id='-eda'></a>
## 🔹 3: Análisis Exploratorio de Datos (EDA)

En esta fase, generamos visualizaciones para validar hipótesis y encontrar correlaciones clave entre las variables y la supervivencia de los pasajeros.

### 📝 3.1 Conteo de Sobrevivientes vs Fallecidos por Género

Evaluamos la proporción de supervivencia segmentada por el género del pasajero.

In [ ]:
# Cálculo de fallecidos
data["Died"] = 1 - data["Survived"]

# Agrupación por género y supervivencia
survival_by_gender = data.groupby("Sex").agg("sum")[["Survived", "Died"]]

# Ploteo de barras apiladas
survival_by_gender.plot(kind="bar", stacked=True, color=["g", "r"], figsize=(10, 6))

plt.title("Sobrevivientes y fallecidos por género", fontsize=16)
plt.xlabel("Género", fontsize=14)
plt.ylabel("Cantidad de Pasajeros", fontsize=14)
plt.xticks(rotation=0)
plt.legend(["Sobrevivientes", "Fallecidos"], fontsize=12)

plt.show()

> **Interpretación:** Las mujeres muestran una tasa de supervivencia significativamente mayor que los hombres, lo que sugiere una prioridad en el rescate basada en el género.

### 📝 3.2 Distribución de Edades por Género y Supervivencia

Analizamos la distribución de edad utilizando un gráfico de violín para observar la densidad de sobrevivientes por rangos etarios.

In [ ]:
# Imputación temporal para visualización
data["Age"] = data["Age"].fillna(data["Age"].median())

plt.figure(figsize=(25, 7))
sns.violinplot(
    x="Sex",
    y="Age",
    hue="Survived",
    data=data,
    split=True,
    palette={0: "r", 1: "g"},
    linewidth=1.5,
)

plt.title("Distribución de edades por género y supervivencia", fontsize=16)
plt.xlabel("Género", fontsize=14)
plt.ylabel("Edad", fontsize=14)
plt.legend(
    title="Sobrevivencia",
    loc="upper right",
    labels=["No sobrevivió", "Sobrevivió"],
    fontsize=12,
)

plt.show()

> **Interpretación:** En los hombres, la edad es un factor determinante: los más jóvenes tuvieron mayor probabilidad de sobrevivir. En las mujeres, la edad no parece impactar de forma tan directa en la supervivencia.

### 📝 3.3 Distribución de Tarifas por Supervivencia

Determinamos si el precio del pasaje (correlacionado con el estatus social) influyó en el rescate.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))

# Histogramas superpuestos
ax.hist(
    data[data["Survived"] == 1]["Fare"],
    bins=50,
    color="g",
    alpha=0.7,
    label="Sobrevivieron",
)
ax.hist(
    data[data["Survived"] == 0]["Fare"],
    bins=50,
    color="r",
    alpha=0.7,
    label="Fallecieron",
)

ax.set_xlabel("Tarifa", fontsize=14)
ax.set_ylabel("Número de Pasajeros", fontsize=14)
ax.set_title("Distribución de Tarifas por Supervivencia", fontsize=16)
ax.legend(loc="upper right", fontsize=12)

plt.show()

> **Interpretación:** Existe una clara tendencia donde los pasajeros con tarifas más bajas tuvieron menos probabilidades de sobrevivir, evidenciando una brecha de clase en el salvamento.

### 📝 3.4 Relación entre Edad, Tarifa y Supervivencia

Utilizamos un gráfico de dispersión para integrar tres dimensiones de análisis y observar patrones multivariados.

In [ ]:
plt.figure(figsize=(14, 7))
ax = plt.subplot()

ax.scatter(
    data[data["Survived"] == 1]["Age"],
    data[data["Survived"] == 1]["Fare"],
    c="green",
    s=data[data["Survived"] == 1]["Fare"],
    label="Sobrevivieron",
    alpha=0.5,
)

ax.scatter(
    data[data["Survived"] == 0]["Age"],
    data[data["Survived"] == 0]["Fare"],
    c="red",
    s=data[data["Survived"] == 0]["Fare"],
    label="Fallecieron",
    alpha=0.5,
)

ax.set_xlabel("Edad", fontsize=14)
ax.set_ylabel("Tarifa", fontsize=14)
ax.set_title("Relación entre Edad, Tarifa y Supervivencia", fontsize=16)
ax.legend()

plt.show()

### 📝 3.5 Tarifa Promedio por Clase (Pclass)

Confirmamos la relación entre la variable categórica Pclass y el costo del pasaje.

In [ ]:
ax = plt.subplot()
ax.set_ylabel("Tarifa Promedio")
data.groupby("Pclass")["Fare"].mean().plot(kind="bar", figsize=(25, 7), ax=ax)
plt.title("Tarifa Promedio por Clase (Pclass)")

plt.show()

---

<a id='-feature-engineering'></a>
## 🔹 4: Feature Engineering (Ingeniería de Características)

En esta sección transformaremos los datos brutos en variables manejables y crearemos nuevas características basadas en el conocimiento del dominio.

### 📝 4.1 Combinación de Datasets

Unificamos los datos de entrenamiento y prueba para garantizar que todas las transformaciones sean uniformes y consistentes.

In [ ]:
def get_combined_data():
    train = pd.read_csv("train.csv")
    test = pd.read_csv("test.csv")

    # Preservar targets
    targets = train.Survived
    train.drop(["Survived"], axis=1, inplace=True)

    # Combinar
    combined = pd.concat([train, test], ignore_index=True)
    combined.reset_index(inplace=True)
    combined.drop(["index", "PassengerId"], inplace=True, axis=1)

    return combined


combined = get_combined_data()
print(f"Dataset combinado creado. Dimensiones: {combined.shape}")

combined.sample(5)

### 📝 4.2 Procesamiento del Título

Extraemos el título de los nombres de los pasajeros para clasificar su estatus social.

In [ ]:
Title_Dictionary = {
    "Capt": "Officer",
    "Col": "Officer",
    "Major": "Officer",
    "Jonkheer": "Royalty",
    "Don": "Royalty",
    "Sir": "Royalty",
    "Dr": "Officer",
    "Rev": "Officer",
    "the Countess": "Royalty",
    "Mme": "Mrs",
    "Mlle": "Miss",
    "Ms": "Mrs",
    "Mr": "Mr",
    "Mrs": "Mrs",
    "Miss": "Miss",
    "Master": "Master",
    "Lady": "Royalty",
}


def get_titles():
    # Extracción y mapeo
    combined["Title"] = combined["Name"].map(
        lambda name: name.split(",")[1].split(".")[0].strip()
    )
    combined["Title"] = combined.Title.map(Title_Dictionary)

    return combined


combined = get_titles()
print(f"Dataset combinado creado. Dimensiones: {combined.shape}")

combined.sample(5)

### 📝 4.3 Procesamiento de Edad (Imputación por Grupos)

Llenamos las edades faltantes utilizando la mediana calculada sobre grupos de Sexo, Clase y Título, evitando sesgos generales.

In [ ]:
grouped_train = combined.iloc[:891].groupby(["Sex", "Pclass", "Title"])["Age"].median()
grouped_median_train = grouped_train.reset_index()[["Sex", "Pclass", "Title", "Age"]]


def fill_age(row):
    condition = (
        (grouped_median_train["Sex"] == row["Sex"])
        & (grouped_median_train["Title"] == row["Title"])
        & (grouped_median_train["Pclass"] == row["Pclass"])
    )
    return grouped_median_train[condition]["Age"].values[0]


def process_age():
    global combined
    combined["Age"] = combined.apply(
        lambda row: fill_age(row) if np.isnan(row["Age"]) else row["Age"], axis=1
    )
    return combined


combined = process_age()
print(f"Dataset combinado creado. Dimensiones: {combined.shape}")

combined.sample(5)

### 📝 4.4 One-Hot Encoding para Títulos

Transformamos la columna categórica `Title` en variables binarias (dummies).

In [ ]:
def process_names():
    global combined
    combined.drop("Name", axis=1, inplace=True)
    titles_dummies = pd.get_dummies(combined["Title"], prefix="Title").astype(int)
    combined = pd.concat([combined, titles_dummies], axis=1)
    combined.drop("Title", axis=1, inplace=True)
    return combined


combined = process_names()
print(f"Dataset combinado creado. Dimensiones: {combined.shape}")

combined.sample(5)

### 📝 4.5 Procesamiento de Tarifas, Embarque y Cabinas

Realizamos imputaciones y codificaciones adicionales para las variables restantes.

In [ ]:
def process_fares():
    global combined
    combined.Fare.fillna(combined.iloc[:891].Fare.mean(), inplace=True)
    return combined


combined = process_fares()
print(f"Dataset combinado creado. Dimensiones: {combined.shape}")

combined.sample(5)

In [ ]:
def process_embarked():
    global combined
    combined.Embarked.fillna("S", inplace=True)
    embarked_dummies = pd.get_dummies(combined["Embarked"], prefix="Embarked").astype(
        int
    )
    combined = pd.concat([combined, embarked_dummies], axis=1)
    combined.drop("Embarked", axis=1, inplace=True)
    return combined


combined = process_embarked()
print(f"Dataset combinado creado. Dimensiones: {combined.shape}")

combined.sample(5)

In [ ]:
def process_cabin():
    global combined
    combined.Cabin.fillna("U", inplace=True)
    combined["Cabin"] = combined["Cabin"].map(lambda c: c[0])
    cabin_dummies = pd.get_dummies(combined["Cabin"], prefix="Cabin").astype(int)
    combined = pd.concat([combined, cabin_dummies], axis=1)
    combined.drop("Cabin", axis=1, inplace=True)
    return combined


combined = process_cabin()
print(f"Dataset combinado creado. Dimensiones: {combined.shape}")

combined.sample(5)

### 📝 4.6 Procesamiento de Tickets

Extraemos prefijos y manejamos los tickets numéricos puros como una categoría especial.

In [ ]:
def cleanTicket(ticket):
    ticket = ticket.replace(".", "").replace("/", "").split()
    ticket = list(filter(lambda t: not t.isdigit(), map(lambda t: t.strip(), ticket)))
    return ticket[0] if len(ticket) > 0 else "XXX"


def process_ticket():
    global combined
    combined["Ticket"] = combined["Ticket"].map(cleanTicket)
    tickets_dummies = pd.get_dummies(combined["Ticket"], prefix="Ticket").astype(int)
    combined = pd.concat([combined, tickets_dummies], axis=1)
    combined.drop("Ticket", inplace=True, axis=1)
    return combined


combined = process_ticket()
print(f"Dataset combinado creado. Dimensiones: {combined.shape}")

combined.sample(5)

### 📝 4.7 Variables de Familia y Grupos de Edad

Generamos variables que resumen el tamaño del núcleo familiar y simplifican el rango etario.

In [ ]:
def process_family():
    global combined
    combined["FamilySize"] = combined["Parch"] + combined["SibSp"] + 1
    combined["Singleton"] = combined["FamilySize"].map(lambda s: 1 if s == 1 else 0)
    combined["SmallFamily"] = combined["FamilySize"].map(
        lambda s: 1 if 2 <= s <= 4 else 0
    )
    combined["LargeFamily"] = combined["FamilySize"].map(lambda s: 1 if 5 <= s else 0)
    return combined


combined = process_family()
print(f"Dataset combinado creado. Dimensiones: {combined.shape}")

combined.sample(5)

In [ ]:
def process_age_groups():
    global combined
    combined["Younger"] = (combined["Age"] < 20).astype(int)
    combined["Adult"] = ((combined["Age"] >= 20) & (combined["Age"] <= 40)).astype(int)
    combined["Elder"] = (combined["Age"] > 40).astype(int)
    return combined


combined = process_age_groups()
print(f"Dataset combinado creado. Dimensiones: {combined.shape}")

combined.sample(5)

---

<a id='-sklearn'></a>
## 🔹 5: Preprocesamiento con Scikit-Learn

En esta sección final, aplicaremos codificación numérica profesional para las variables categóricas restantes.

### 📝 5.1 Codificación de Sexo con LabelEncoder

Para asegurar un flujo de datos limpio y uniforme, utilizamos `LabelEncoder` para transformar el Sexo y la Clase del Pasajero (Pclass) en valores numéricos. 

**Mantenimiento de Consistencia:** Siguiendo las directrices profesionales, se ha sustituido el método de dummies para Pclass por una codificación basada en Scikit-Learn.

In [ ]:
def process_scikit_label_encoding():
    global combined
    le = LabelEncoder()
    # Codificación de Sexo
    combined["Sex"] = le.fit_transform(combined["Sex"])
    return combined


combined = process_scikit_label_encoding()
print(f"Dataset combinado creado. Dimensiones: {combined.shape}")

combined.sample(5)

<a id='-sklearn'></a>
## 🔹 6: Modeling and Prediction

En esta fase final, implementaremos un pipeline de aprendizaje automático para predecir la supervivencia de los pasajeros. El proceso abarca desde la preparación de los conjuntos de datos hasta la selección de las variables más influyentes y la evaluación de múltiples algoritmos para identificar el modelo con mayor capacidad predictiva.

### 📝 6.1 Preparación de Herramientas y Datos

Comenzamos importando los módulos necesarios de Scikit-Learn. Utilizaremos clasificadores populares, herramientas para la validación cruzada y utilidades para la selección de características.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.feature_selection import SelectKBest, SelectFromModel
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.metrics import accuracy_score
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

Definimos una función para recuperar los conjuntos de entrenamiento y prueba a partir del dataframe `combined`. Es crucial separar nuevamente los datos para evitar que el modelo 'conozca' las etiquetas del conjunto de prueba durante el entrenamiento (*Data Leakage*).

In [ ]:
def recover_train_test_target():
    global combined
    targets = pd.read_csv("train.csv", usecols=["Survived"])["Survived"].values
    train = combined.iloc[:891]
    test = combined.iloc[891:]
    return train, test, targets

In [ ]:
train, test, targets = recover_train_test_target()

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Targets:", len(targets))

### 📝 6.2 Evaluación de Relevancia (Feature Importance)

El primer paso técnico es evaluar la importancia de cada característica. Utilizamos un modelo de Bosque Aleatorio (`RandomForestClassifier`) debido a su capacidad intrínseca para cuantificar cuánto contribuye cada variable a la reducción de la impureza en los nodos de decisión. Graficar la importancia nos permite visualizar qué factores (como la edad, el sexo o la tarifa) son realmente determinantes en la predicción.

In [ ]:
clf = RandomForestClassifier(n_estimators=50, max_features="sqrt", random_state=42)

clf.fit(train, targets)

features = pd.DataFrame()
features["feature"] = train.columns
features["importance"] = clf.feature_importances_
features.sort_values(by=["importance"], ascending=True, inplace=True)
features.set_index("feature", inplace=True)

plt.figure(figsize=(20, 20))
features.plot(kind="barh", figsize=(20, 20))

# línea indicando la media de importancia
mean_imp = features["importance"].mean()
plt.axvline(mean_imp, linestyle="--")
plt.title("Feature Importance (RandomForest)")
plt.show()

Basándonos en el análisis anterior, realizamos una reducción de dimensionalidad utilizando `SelectFromModel`. Al conservar solo aquellas características cuya importancia es superior al promedio, eliminamos el ruido y mejoramos tanto la eficiencia computacional como la capacidad de generalización del modelo.

In [ ]:
model = SelectFromModel(clf, prefit=True)

train_reduced = model.transform(train)
test_reduced = model.transform(test)

print("Train reduced:", train_reduced.shape)
print("Test reduced:", test_reduced.shape)

### 📝 6.3 Comparación de Modelos y Validación Cruzada

Para encontrar el algoritmo más robusto, comparamos cuatro modelos distintos: Regresión Logística (estándar y con validación cruzada interna), Bosque Aleatorio y Gradient Boosting. Aplicamos validación cruzada de 5 pliegues para obtener una estimación del *accuracy* que sea estadísticamente significativa y menos dependiente de una partición específica de los datos.

In [ ]:
def compute_score(clf, X, y, scoring="accuracy"):
    xval = cross_val_score(clf, X, y, cv=5, scoring=scoring)
    return np.mean(xval)


logreg = LogisticRegression(max_iter=500)
logreg_cv = LogisticRegressionCV(max_iter=500)
rf = RandomForestClassifier()
gboost = GradientBoostingClassifier()

models = [
    ("LogisticRegression", logreg),
    ("LogisticRegressionCV", logreg_cv),
    ("RandomForest", rf),
    ("GradientBoosting", gboost),
]

results = {}

for name, model in models:
    print("Cross-validation of:", name)
    score = compute_score(model, X=train_reduced, y=targets, scoring="accuracy")
    results[name] = score
    print("CV score =", score)
    print("************")

results

Una vez obtenidos los resultados, seleccionamos el modelo con el mejor desempeño promedio. Generalmente, algoritmos de ensamble como `GradientBoostingClassifier` tienden a ofrecer resultados superiores en este tipo de datasets tabulares competitivos.

In [ ]:
best_model_name = max(results, key=results.get)
best_model_score = results[best_model_name]

print("Best model:", best_model_name)
print("Score:", best_model_score)

# entrenar el mejor modelo con todos los datos
if best_model_name == "GradientBoosting":
    best_model = GradientBoostingClassifier()
elif best_model_name == "RandomForest":
    best_model = RandomForestClassifier()
elif best_model_name == "LogisticRegression":
    best_model = LogisticRegression(max_iter=500)
else:
    best_model = LogisticRegressionCV(max_iter=500)

best_model.fit(train_reduced, targets)

### 📝 6.4 Implementación de la Función de Predicción

Encapsulamos la lógica de inferencia en una función personalizada. Esta función se encarga de transformar los datos de entrada de un nuevo pasajero al mismo formato exacto (columnas, orden y codificación) que el modelo espera, asegurando la consistencia entre el entrenamiento y la aplicación práctica.

In [ ]:
def predict_person(model, model_selector, **kwargs):
    """
    Recibe los datos de un pasajero (kwargs), los transforma al mismo
    formato que el dataframe 'train' y aplica el modelo.
    """
    global combined  # solo esto, nada más

    # convertir a DataFrame temporal
    df = pd.DataFrame([kwargs])

    # asegurar que existan todas las columnas del training
    for col in train.columns:
        if col not in df.columns:
            df[col] = 0

    # reordenar columnas para que coincidan con el training
    df = df[train.columns]

    # reducir features con el mismo selector usado en train
    df_reduced = model_selector.transform(df)

    # predecir
    prediction = model.predict(df_reduced)[0]
    return prediction

In [ ]:
model_selector = SelectFromModel(clf, prefit=True)

Probamos nuestro sistema con un caso hipotético para validar que el pipeline completo, desde el preprocesamiento de las variables hasta la decisión del modelo, funciona correctamente.

In [ ]:
jorge = predict_person(
    model=best_model,
    model_selector=model_selector,
    Pclass=2,
    Sex_male=1,
    Sex_female=0,
    Age=35,
    SibSp=0,
    Parch=0,
    Fare=20,
    Embarked_S=1,
    Embarked_C=0,
    Embarked_Q=0,
    Title_Mr=1,
)

print("¿Jorge sobreviviría?:", "Sí" if jorge == 1 else "No")